# hrm quickstart

Book GPUs on the shared box from Python. This notebook:

1. installs the `hrm` SDK,
2. shows current GPU usage and the booking calendar as pandas DataFrames,
3. places a small booking (1 GiB, today only) and cancels it straight away.

The outputs come from one run against the live server; user names and IPs in them are anonymised.

## Install

Installs the SDK with the optional pandas extra. Until the repo is public/pushed, install a local checkout instead: `pip install -e "..[pandas]"` from this folder.

Set `HRM_SKIP_INSTALL=1` to skip this cell when the SDK is already installed.

In [1]:
import os

if not os.environ.get("HRM_SKIP_INSTALL"):
    %pip install "hnee-resource-monitoring[pandas] @ git+https://github.com/cwinkelmann/hnee-resource-monitoring-sdk"

## Connect

Say who you are once. Use your carrot account name; `HRM_USER` and `HRM_URL` in the environment work too.

In [2]:
import hrm

hrm.connect(user="cwinkelmann")
print("hrm", hrm.__version__)

hrm 0.1.0


## Current usage

One row per GPU: memory in use, memory free after bookings, utilisation, power, who has processes on it, and the active bookings.

In [3]:
hrm.usage().to_frame()

,gpu,used_gib,total_gib,free_gib,util_pct,power_w,users,booked
0,0,0.0,79.6,0.0,0,65.90,,cwinkelmann 79.6 GiB until Wed 09:00
1,1,0.0,79.6,49.6,0,66.69,,cwinkelmann 30 GiB until Wed 16:18
2,2,0.0,79.6,79.6,0,70.37,,
3,3,0.0,79.6,79.6,0,69.49,,
4,4,0.0,79.6,79.6,0,66.99,,
5,5,0.0,79.6,79.6,0,71.69,,
6,6,0.0,79.6,39.6,0,69.75,,carol 40 GiB until Fri 18:00
7,7,0.0,79.6,39.6,0,70.38,,carol 40 GiB until Fri 18:00


## Bookings

Active bookings from 7 days back to 14 days ahead (`hrm.bookings(days_ahead=14, days_back=7)`).

In [4]:
hrm.bookings().to_frame()

,id,user,gpu,vram_gib,start,end,note,kind,cancelled
0,3,carol,6,40.0,2026-10-06 14:57:00+02:00,2026-10-09 18:00:00+02:00,NaN,calendar,False
1,4,carol,7,40.0,2026-10-06 14:57:00+02:00,2026-10-09 18:00:00+02:00,NaN,calendar,False
2,7,cwinkelmann,0,79.6,2026-10-06 16:06:30.818990+02:00,2026-10-07 09:00:00+02:00,quick booking,quick,False
3,10,cwinkelmann,1,30.0,2026-10-06 16:18:00+02:00,2026-10-07 16:18:00+02:00,NaN,calendar,False


## Book a GPU

`hrm.book(days=["2026-10-06", "2026-10-07"], memory=48)` books 48 GiB of GPU memory on one GPU for those two days (Europe/Berlin calendar days, starting now if today is in the list). `memory` is in GiB; leave it out to book the whole card. The SDK picks the lowest-numbered GPU that has enough free memory for every day; pass `gpu=3` to choose one yourself.

This demo books only **1 GiB for the rest of today** and cancels it in the next cell.

In [5]:
from datetime import datetime
from zoneinfo import ZoneInfo

result = hrm.book(days=[datetime.now(ZoneInfo("Europe/Berlin")).date().isoformat()], memory=1, note="sdk quickstart")
print(result)

booked GPU 1, 1 GiB, Tue 06 Oct 17:09 → Wed 07 Oct 00:00 (Europe/Berlin), id 11


## Cancel it

`hrm.cancel()` takes a booking or its id. Cancelling twice is harmless.

In [6]:
cancelled = hrm.cancel(result.bookings[0])
print(cancelled)
print("cancelled:", cancelled.cancelled)

#11 GPU 1 · 1.0 GiB · cwinkelmann · Tue 06 Oct 17:09 → Wed 07 Oct 00:00
cancelled: True


## Check

`hrm.bookings()` hides cancelled bookings; `include_cancelled=True` shows them.

In [7]:
df = hrm.bookings(include_cancelled=True).to_frame()
df[df["id"].isin(result.ids)]

,id,user,gpu,vram_gib,start,end,note,kind,cancelled
9,11,cwinkelmann,1,1.0,2026-10-06 17:09:14.786929+02:00,2026-10-07 00:00:00+02:00,sdk quickstart,calendar,True
